In [ ]:
%pip install requests pandas

In [ ]:
pip install requests pandas

In [ ]:
!python -m pip install requests pandas

In [ ]:
import requests

keywords = [
    "Parag Parikh Flexi Cap", "HDFC Flexi Cap", "SBI Bluechip",
    "Nippon India Small Cap", "Axis Midcap", "UTI Nifty 50 Index",
    "ICICI Prudential Bluechip", "Kotak Emerging Equity",
    "Mirae Asset Large Cap", "HDFC Corporate Bond",
]

for k in keywords:
    r = requests.get("https://api.mfapi.in/mf/search", params={"q": k})
    print("====", k)
    for s in r.json()[:8]:
        print(s["schemeCode"], s["schemeName"])

==== Parag Parikh Flexi Cap
122640 Parag Parikh Flexi Cap Fund - Regular Plan - Growth
122639 Parag Parikh Flexi Cap Fund - Direct Plan - Growth
153964 Parag Parikh Flexi Cap Fund - Direct Plan - Monthly IDCW Payout
153965 Parag Parikh Flexi Cap Fund - Regular Plan - Monthly IDCW Payout
==== HDFC Flexi Cap
101762 HDFC Flexi Cap Fund - Regular Plan - Growth Option
101763 HDFC Flexi Cap Fund - Regular Plan - IDCW Option
118954 HDFC Flexi Cap Fund - Direct Plan - IDCW Option
118955 HDFC Flexi Cap Fund - Direct Plan - Growth Option
==== SBI Bluechip
==== Nippon India Small Cap
113177 Nippon India Small Cap Fund - Regular Plan - Growth Option
113178 Nippon India Small Cap Fund - Regular Plan - Bonus Option
113179 Nippon India Small Cap Fund - Regular Plan - IDCW Option
118775 Nippon India Small Cap Fund - Direct Plan - IDCW Option
118777 Nippon India Small Cap Fund - Direct Plan - Bonus Option
118778 Nippon India Small Cap Fund - Direct Plan - Growth Option
==== Axis Midcap
114564 Axis Midc

In [ ]:
import requests, pandas as pd, time

keywords = [
    "Parag Parikh Flexi Cap", "HDFC Flexi Cap", "SBI Bluechip",
    "Nippon India Small Cap", "Axis Midcap", "UTI Nifty 50 Index",
    "ICICI Prudential Bluechip", "Kotak Emerging Equity",
    "Mirae Asset Large Cap", "HDFC Corporate Bond",
]

# Step 1: find funds (Growth plans only, up to 2 per search)
codes = {}
for k in keywords:
    try:
        res = requests.get("https://api.mfapi.in/mf/search", params={"q": k}).json()
        picked = 0
        for s in res:
            n = s["schemeName"]
            if "Growth" in n and "IDCW" not in n and "Dividend" not in n:
                codes[s["schemeCode"]] = n
                picked += 1
                if picked == 2:
                    break
    except Exception as e:
        print("search failed for", k, e)
    time.sleep(1)

print("Funds found:", len(codes))
for c, n in codes.items():
    print(c, n)

# Step 2: download the full NAV history for each fund
frames = []
for c in codes:
    try:
        j = requests.get(f"https://api.mfapi.in/mf/{c}").json()
        df = pd.DataFrame(j["data"])
        df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")
        df["nav"] = df["nav"].astype(float)
        df["scheme_code"] = c
        df["scheme_name"] = j["meta"]["scheme_name"]
        df["fund_house"] = j["meta"].get("fund_house")
        df["category"] = j["meta"].get("scheme_category")
        frames.append(df)
    except Exception as e:
        print("download failed for", c, e)
    time.sleep(1)

data = pd.concat(frames)
data.to_csv("mf_nav.csv", index=False)
print(data.shape)
print(data.columns.tolist())
data.head()

Funds found: 18
122640 Parag Parikh Flexi Cap Fund - Regular Plan - Growth
122639 Parag Parikh Flexi Cap Fund - Direct Plan - Growth
101762 HDFC Flexi Cap Fund - Regular Plan - Growth Option
118955 HDFC Flexi Cap Fund - Direct Plan - Growth Option
113177 Nippon India Small Cap Fund - Regular Plan - Growth Option
118778 Nippon India Small Cap Fund - Direct Plan - Growth Option
114564 Axis Midcap Fund - Regular Plan - Growth Option
120505 Axis Midcap Fund - Direct Plan - Growth Option
100822 UTI Nifty 50 Index Fund - Regular Plan - Growth
120716 UTI Nifty 50 Index Fund - Direct Plan - Growth
120186 ICICI Prudential US Bluechip Equity Fund - Direct Plan - Growth
117620 ICICI Prudential US Bluechip Equity Fund - Regular Plan - Growth
119779 Kotak Global Emerging Market Overseas Equity Active FOF - Direct Plan - Growth
106441 Kotak Global Emerging Market Overseas Equity Active FOF - Regular Plan - Growth
107578 Mirae Asset Large Cap Fund - Regular Plan - Growth
118825 Mirae Asset Large Cap 

,date,nav,scheme_code,scheme_name,fund_house,category
0,2026-10-27,81.2702,122640,Parag Parikh Flexi Cap Fund - Regular Plan - G...,PPFAS Mutual Fund,Equity Scheme - Flexi Cap Fund
1,2026-10-26,81.2932,122640,Parag Parikh Flexi Cap Fund - Regular Plan - G...,PPFAS Mutual Fund,Equity Scheme - Flexi Cap Fund
2,2026-10-23,80.9965,122640,Parag Parikh Flexi Cap Fund - Regular Plan - G...,PPFAS Mutual Fund,Equity Scheme - Flexi Cap Fund
3,2026-10-22,81.3978,122640,Parag Parikh Flexi Cap Fund - Regular Plan - G...,PPFAS Mutual Fund,Equity Scheme - Flexi Cap Fund
4,2026-10-21,81.8866,122640,Parag Parikh Flexi Cap Fund - Regular Plan - G...,PPFAS Mutual Fund,Equity Scheme - Flexi Cap Fund


In [ ]:
data.shape

(67914, 6)

In [ ]:
data["scheme_name"].nunique()

18

In [ ]:
data["date"].min()

Timestamp('2006-04-03 00:00:00')

In [ ]:
data["date"].max()

Timestamp('2026-10-28 00:00:00')

In [ ]:
data.head()

,date,nav,scheme_code,scheme_name,fund_house,category
0,2026-10-27,81.2702,122640,Parag Parikh Flexi Cap Fund - Regular Plan - G...,PPFAS Mutual Fund,Equity Scheme - Flexi Cap Fund
1,2026-10-26,81.2932,122640,Parag Parikh Flexi Cap Fund - Regular Plan - G...,PPFAS Mutual Fund,Equity Scheme - Flexi Cap Fund
2,2026-10-23,80.9965,122640,Parag Parikh Flexi Cap Fund - Regular Plan - G...,PPFAS Mutual Fund,Equity Scheme - Flexi Cap Fund
3,2026-10-22,81.3978,122640,Parag Parikh Flexi Cap Fund - Regular Plan - G...,PPFAS Mutual Fund,Equity Scheme - Flexi Cap Fund
4,2026-10-21,81.8866,122640,Parag Parikh Flexi Cap Fund - Regular Plan - G...,PPFAS Mutual Fund,Equity Scheme - Flexi Cap Fund


In [ ]:
data.groupby("category")["scheme_name"].nunique()

,scheme_name
category,
Debt Scheme - Corporate Bond Fund,2
Equity Scheme - Flexi Cap Fund,4
Equity Scheme - Mid Cap Fund,2
Equity Scheme - Sectoral/ Thematic,2
Equity Scheme - Small Cap Fund,2
Equity Schemes - Large Cap Fund,2
Other Scheme - Index Funds,2
Overseas Fund of Funds - Fund of Funds investing overseas,2


In [ ]:
data.groupby("scheme_name")["date"].min()

,date
scheme_name,
Axis Midcap Fund - Direct Plan - Growth Option,2013-01-02
Axis Midcap Fund - Regular Plan - Growth Option,2011-02-24
HDFC Corporate Bond Fund - Direct Plan - Growth Option,2013-01-01
HDFC Corporate Bond Fund - Regular Plan - Growth Option,2010-06-29
HDFC Flexi Cap Fund - Direct Plan - Growth Option,2013-01-01
HDFC Flexi Cap Fund - Regular Plan - Growth Option,2006-04-03
ICICI Prudential US Bluechip Equity Fund - Direct Plan - Growth,2013-01-03
ICICI Prudential US Bluechip Equity Fund - Regular Plan - Growth,2012-07-09
Kotak Global Emerging Market Overseas Equity Active FOF - Direct Plan - Growth,2013-01-07


In [ ]:
data.isna().sum()

,0
date,0
nav,0
scheme_code,0
scheme_name,0
fund_house,0
category,0


In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data_c = data[data["date"] >= "2013-05-28"]
data_c.shape

(58908, 6)

In [ ]:
data_c = data_c.sort_values("date")
start = data_c.groupby("scheme_name")["nav"].first()
end = data_c.groupby("scheme_name")["nav"].last()
summary = pd.DataFrame({"start_nav": start, "end_nav": end})
summary

,start_nav,end_nav
scheme_name,,
Axis Midcap Fund - Direct Plan - Growth Option,12.9000,136.1700
Axis Midcap Fund - Regular Plan - Growth Option,12.8600,115.6700
HDFC Corporate Bond Fund - Direct Plan - Growth Option,13.0857,34.8782
HDFC Corporate Bond Fund - Regular Plan - Growth Option,13.0817,34.0192
HDFC Flexi Cap Fund - Direct Plan - Growth Option,288.5900,2166.2880
HDFC Flexi Cap Fund - Regular Plan - Growth Option,287.8210,1965.2550
ICICI Prudential US Bluechip Equity Fund - Direct Plan - Growth,12.5400,91.1200
ICICI Prudential US Bluechip Equity Fund - Regular Plan - Growth,12.5000,80.7600
Kotak Global Emerging Market Overseas Equity Active FOF - Direct Plan - Growth,12.9530,40.3660


In [ ]:
summary['start_date'] = data_c.groupby('scheme_name')["date"].first()
summary["end_date"] = data_c.groupby("scheme_name")["date"].last()
summary["years"]=(summary["end_date"]-summary["start_date"]).dt.days / 365.25
summary["cagr_pct"]=((summary["end_nav"]/ summary["start_nav"])**(1/summary["years"])-1)*100
summary.round(2)

,start_nav,end_nav,start_date,end_date,years,cagr_pct
scheme_name,,,,,,
Axis Midcap Fund - Direct Plan - Growth Option,12.90,136.17,2013-05-28,2026-10-28,13.42,19.20
Axis Midcap Fund - Regular Plan - Growth Option,12.86,115.67,2013-05-28,2026-10-28,13.42,17.79
HDFC Corporate Bond Fund - Direct Plan - Growth Option,13.09,34.88,2013-05-28,2026-10-28,13.42,7.58
HDFC Corporate Bond Fund - Regular Plan - Growth Option,13.08,34.02,2013-05-28,2026-10-28,13.42,7.38
HDFC Flexi Cap Fund - Direct Plan - Growth Option,288.59,2166.29,2013-05-28,2026-10-28,13.42,16.21
HDFC Flexi Cap Fund - Regular Plan - Growth Option,287.82,1965.26,2013-05-28,2026-10-28,13.42,15.39
ICICI Prudential US Bluechip Equity Fund - Direct Plan - Growth,12.54,91.12,2013-05-28,2026-10-28,13.42,15.93
ICICI Prudential US Bluechip Equity Fund - Regular Plan - Growth,12.50,80.76,2013-05-28,2026-10-28,13.42,14.92
Kotak Global Emerging Market Overseas Equity Active FOF - Direct Plan - Growth,12.95,40.37,2013-05-28,2026-10-28,13.42,8.84


In [ ]:
summary = summary.reset_index()
summary["fund"] = summary["scheme_name"].str.split(" - ").str[0]
summary["plan"] = summary["scheme_name"].str.contains("Direct").map({True: "Direct", False: "Regular"})
pivot = summary.pivot(index="fund", columns="plan", values="cagr_pct")
pivot["gap"] = pivot["Direct"] - pivot["Regular"]
pivot.round(2)

plan,Direct,Regular,gap
fund,,,
Axis Midcap Fund,19.20,17.79,1.41
HDFC Corporate Bond Fund,7.58,7.38,0.20
HDFC Flexi Cap Fund,16.21,15.39,0.82
ICICI Prudential US Bluechip Equity Fund,15.93,14.92,1.01
Kotak Global Emerging Market Overseas Equity Active FOF,8.84,8.14,0.70
Mirae Asset Large Cap Fund,14.86,13.77,1.09
Nippon India Small Cap Fund,25.12,23.99,1.13
Parag Parikh Flexi Cap Fund,17.73,16.90,0.82
UTI Nifty 50 Index Fund,11.30,11.18,0.12


In [ ]:
data_c = data_c.sort_values(["scheme_name", "date"])
data_c["daily_return"] = data_c.groupby("scheme_name")["nav"].pct_change()
data_c.head()

,date,nav,scheme_code,scheme_name,fund_house,category,daily_return
3309,2013-05-28,12.90,120505,Axis Midcap Fund - Direct Plan - Growth Option,Axis Mutual Fund,Equity Scheme - Mid Cap Fund,NaN
3308,2013-05-29,12.84,120505,Axis Midcap Fund - Direct Plan - Growth Option,Axis Mutual Fund,Equity Scheme - Mid Cap Fund,-0.004651
3307,2013-05-30,12.86,120505,Axis Midcap Fund - Direct Plan - Growth Option,Axis Mutual Fund,Equity Scheme - Mid Cap Fund,0.001558
3306,2013-05-31,12.74,120505,Axis Midcap Fund - Direct Plan - Growth Option,Axis Mutual Fund,Equity Scheme - Mid Cap Fund,-0.009331
3305,2013-06-03,12.79,120505,Axis Midcap Fund - Direct Plan - Growth Option,Axis Mutual Fund,Equity Scheme - Mid Cap Fund,0.003925


In [ ]:
data_c["peak"] = data_c.groupby("scheme_name")["nav"].cummax()
data_c["drawdown"] = data_c["nav"] / data_c["peak"] - 1
mdd = data_c.groupby("scheme_name")["drawdown"].min() * 100
mdd.sort_values().round(1)

,drawdown
scheme_name,
Nippon India Small Cap Fund - Regular Plan - Growth Option,-49.7
Nippon India Small Cap Fund - Direct Plan - Growth Option,-48.6
HDFC Flexi Cap Fund - Regular Plan - Growth Option,-42.1
HDFC Flexi Cap Fund - Direct Plan - Growth Option,-41.8
UTI Nifty 50 Index Fund - Regular Plan - Growth,-38.4
UTI Nifty 50 Index Fund - Direct Plan - Growth,-38.4
Mirae Asset Large Cap Fund - Regular Plan - Growth,-37.6
Mirae Asset Large Cap Fund - Direct Plan - Growth,-37.4
Kotak Global Emerging Market Overseas Equity Active FOF - Regular Plan - Growth,-32.6


In [ ]:
trough = data_c.sort_values("drawdown").groupby("scheme_name").first()[["date", "drawdown"]]
trough.sort_values("date")

,date,drawdown
scheme_name,,
HDFC Corporate Bond Fund - Direct Plan - Growth Option,2013-08-19,-0.040034
HDFC Corporate Bond Fund - Regular Plan - Growth Option,2013-08-19,-0.040132
Axis Midcap Fund - Direct Plan - Growth Option,2020-03-23,-0.293625
Mirae Asset Large Cap Fund - Regular Plan - Growth,2020-03-23,-0.375673
Mirae Asset Large Cap Fund - Direct Plan - Growth,2020-03-23,-0.374417
UTI Nifty 50 Index Fund - Direct Plan - Growth,2020-03-23,-0.384179
ICICI Prudential US Bluechip Equity Fund - Regular Plan - Growth,2020-03-23,-0.246925
UTI Nifty 50 Index Fund - Regular Plan - Growth,2020-03-23,-0.384262
HDFC Flexi Cap Fund - Regular Plan - Growth Option,2020-03-23,-0.420598


In [ ]:
recovery = {}
for name, g in data_c.groupby("scheme_name"):
    t_date = trough.loc[name, "date"]
    after = g[(g["date"] > t_date) & (g["drawdown"] == 0)]
    recovery[name] = after["date"].min()

rec = pd.DataFrame({"trough_date": trough["date"], "recovery_date": pd.Series(recovery)})
rec["days_to_recover"] = (rec["recovery_date"] - rec["trough_date"]).dt.days
rec.sort_values("days_to_recover")

,trough_date,recovery_date,days_to_recover
HDFC Corporate Bond Fund - Regular Plan - Growth Option,2013-08-19,2013-10-08,50
HDFC Corporate Bond Fund - Direct Plan - Growth Option,2013-08-19,2013-10-08,50
ICICI Prudential US Bluechip Equity Fund - Direct Plan - Growth,2020-03-23,2020-06-03,72
ICICI Prudential US Bluechip Equity Fund - Regular Plan - Growth,2020-03-23,2020-06-03,72
Parag Parikh Flexi Cap Fund - Regular Plan - Growth,2020-03-24,2020-07-07,105
Parag Parikh Flexi Cap Fund - Direct Plan - Growth,2020-03-24,2020-07-07,105
Axis Midcap Fund - Direct Plan - Growth Option,2020-03-23,2020-08-26,156
Axis Midcap Fund - Regular Plan - Growth Option,2020-03-23,2020-09-16,177
Mirae Asset Large Cap Fund - Direct Plan - Growth,2020-03-23,2020-11-06,228
Mirae Asset Large Cap Fund - Regular Plan - Growth,2020-03-23,2020-11-09,231


In [ ]:
table = summary.set_index("scheme_name")[["cagr_pct"]].copy()
table["max_drawdown_pct"] = mdd
table["days_to_recover"] = rec["days_to_recover"]
table.round(1).sort_values("cagr_pct", ascending=False)

,cagr_pct,max_drawdown_pct,days_to_recover
scheme_name,,,
Nippon India Small Cap Fund - Direct Plan - Growth Option,25.1,-48.6,265
Nippon India Small Cap Fund - Regular Plan - Growth Option,24.0,-49.7,281
Axis Midcap Fund - Direct Plan - Growth Option,19.2,-29.4,156
Axis Midcap Fund - Regular Plan - Growth Option,17.8,-29.4,177
Parag Parikh Flexi Cap Fund - Direct Plan - Growth,17.7,-31.2,105
Parag Parikh Flexi Cap Fund - Regular Plan - Growth,16.9,-31.3,105
HDFC Flexi Cap Fund - Direct Plan - Growth Option,16.2,-41.8,261
ICICI Prudential US Bluechip Equity Fund - Direct Plan - Growth,15.9,-24.6,72
HDFC Flexi Cap Fund - Regular Plan - Growth Option,15.4,-42.1,266


In [ ]:
from google.colab import files
data.to_csv("mf_nav.csv", index=False)
table.round(2).reset_index().to_csv("fund_summary.csv", index=False)
files.download("mf_nav.csv")
files.download("fund_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
pivot.round(2)

plan,Direct,Regular,gap
fund,,,
Axis Midcap Fund,19.20,17.79,1.41
HDFC Corporate Bond Fund,7.58,7.38,0.20
HDFC Flexi Cap Fund,16.21,15.39,0.82
ICICI Prudential US Bluechip Equity Fund,15.93,14.92,1.01
Kotak Global Emerging Market Overseas Equity Active FOF,8.84,8.14,0.70
Mirae Asset Large Cap Fund,14.86,13.77,1.09
Nippon India Small Cap Fund,25.12,23.99,1.13
Parag Parikh Flexi Cap Fund,17.73,16.90,0.82
UTI Nifty 50 Index Fund,11.30,11.18,0.12
